# Malawi Landslide Dataset — PlanetScope Exact-Patch Rebuilder

This notebook reconstructs the PlanetScope-derived **RGB, NIR and NDVI** patches omitted from the open dataset.

For every manifest patch it verifies a real `PSScene` catalog intersection and uses the Planet Orders API **Clip** tool, so it requests only the required patch area rather than intentionally downloading full scenes. Planet supports Clip for the PSScene Surface Reflectance bundles used here.

**RGB and NIR now reproduce the original dataset preprocessing exactly:** after the final 128 × 128 raw/SR patch is placed on the target grid, each band is independently stretched using valid pixels (`band > 0`), the **2nd and 98th percentiles**, clipping to `[0,1]`, multiplication by 255, and `astype(uint8)` truncation.

Outputs preserve the exact manifest georeferencing: **EPSG:32736, 3 m, 128 × 128, exact bounds/transform**. Users provide their own authorized Planet API key.


In [ ]:
# ============================================================
# Malawi Landslide Dataset — PlanetScope Exact-Patch Rebuilder
# SINGLE-CELL PUBLIC NOTEBOOK
#
# Goal:
#   Rebuild ONLY the PlanetScope optical footprints required by
#   the public dataset. Full PlanetScope scenes are not downloaded.
#
# For every CSV row the notebook:
#   1. Uses the exact 128x128 / 3 m patch footprint from the CSV.
#   2. Converts that footprint from EPSG:32736 to WGS84.
#   3. Checks the Planet Data API for PSScene catalog items that
#      REALLY intersect that footprint on the required post-event
#      date(s), instead of blindly trusting a preliminary item ID.
#   4. Prefers IDs recorded in the manifest when they truly
#      intersect; otherwise uses the actual intersecting catalog
#      item(s) from the required acquisition date.
#   5. Orders ONLY the patch AOI with the Planet Orders Clip tool.
#   6. If several source items cover the patch, requests them
#      together and uses Planet's Composite tool.
#   7. Reprojects the delivered clipped SR raster onto the EXACT
#      original dataset grid.
#   8. Saves RGB, NIR and NDVI as uint8 GeoTIFFs with exact
#      georeferencing and original patch filenames.
#
# Users need their own authorized Planet API key and imagery access.
# ============================================================

# ======================= USER SETTINGS ========================
# Fill these TWO paths only:
CSV_PATH = r""              # e.g. r"C:\...\PlanetScope_post_source_per_patch.csv"
OUTPUT_DATASET_ROOT = r""   # e.g. r"C:\...\Malawi_Landslide_Dataset"

# Optional cache. Leave blank for an automatic cache folder.
CACHE_DIR = r""

OVERWRITE = False

# Search policy:
# Exact post_date is searched first. If no catalog item intersects,
# dates listed in all_intersecting_dates (when present) are also used.
# Only if still empty, SEARCH_DAY_TOLERANCE allows +/- N days.
SEARCH_DAY_TOLERANCE = 0

# RGB/NIR uint8 conversion exactly follows the original dataset code:
# independent per-band 2nd-98th percentile stretch on each final 128x128 patch.

# Exact target grid used by this dataset.
TARGET_CRS = "EPSG:32736"
TARGET_RESOLUTION = 3.0
TARGET_WIDTH = 128
TARGET_HEIGHT = 128
# ============================================================

import sys, subprocess, importlib.util

_packages = {
    "requests": "requests",
    "pandas": "pandas",
    "numpy": "numpy",
    "rasterio": "rasterio",
    "tqdm": "tqdm",
    "pyproj": "pyproj",
}
_missing = [pip for mod, pip in _packages.items() if importlib.util.find_spec(mod) is None]
if _missing:
    print("Installing missing packages:", ", ".join(_missing))
    subprocess.check_call([sys.executable, "-m", "pip", "install", *_missing])

from pathlib import Path
from getpass import getpass
from datetime import datetime, timedelta
import ast, json, re, time, uuid
import requests
import numpy as np
import pandas as pd
import rasterio
from rasterio.transform import from_origin
from rasterio.warp import reproject, Resampling
from pyproj import Transformer
from tqdm.auto import tqdm

# ------------------------- PATHS ------------------------------
if not CSV_PATH.strip():
    raise ValueError("Fill CSV_PATH at the top of the cell.")
if not OUTPUT_DATASET_ROOT.strip():
    raise ValueError("Fill OUTPUT_DATASET_ROOT at the top of the cell.")

CSV_PATH = Path(CSV_PATH)
OUTPUT_DATASET_ROOT = Path(OUTPUT_DATASET_ROOT)
CACHE_DIR = Path(CACHE_DIR) if CACHE_DIR.strip() else OUTPUT_DATASET_ROOT / "_planetscope_patch_cache"
LOG_PATH = OUTPUT_DATASET_ROOT / "PlanetScope_reconstruction_log.csv"
PROVENANCE_PATH = OUTPUT_DATASET_ROOT / "PlanetScope_verified_patch_provenance.csv"

if not CSV_PATH.exists():
    raise FileNotFoundError(f"CSV not found:\n{CSV_PATH}")

OUTPUT_DATASET_ROOT.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)
for split in ("train", "val", "test"):
    for modality in ("RGB", "NIR", "NDVI"):
        (OUTPUT_DATASET_ROOT / split / modality).mkdir(parents=True, exist_ok=True)

DATA_QUICK_SEARCH = "https://api.planet.com/data/v1/quick-search"
ORDERS_URL = "https://api.planet.com/compute/ops/orders/v2"
ITEM_TYPE = "PSScene"

# ------------------------- MANIFEST ---------------------------
df = pd.read_csv(CSV_PATH)

required = [
    "split", "patch_file", "left", "bottom", "right", "top",
    "post_date", "post_item_id"
]
missing_cols = [c for c in required if c not in df.columns]
if missing_cols:
    raise ValueError("CSV missing required columns: " + ", ".join(missing_cols))

df["split"] = df["split"].astype(str).str.strip().str.lower()
df["post_item_id"] = df["post_item_id"].astype(str).str.strip()

if not df["split"].isin(["train", "val", "test"]).all():
    raise ValueError("Invalid split value in CSV.")

print("=" * 88)
print("MALAWI LANDSLIDE DATASET — PLANETSCOPE EXACT-PATCH REBUILDER")
print("=" * 88)
print("Manifest :", CSV_PATH)
print("Output   :", OUTPUT_DATASET_ROOT)
print("Cache    :", CACHE_DIR)
print("Patches  :", f"{len(df):,}")
print("Manifest item IDs:", f"{df['post_item_id'].nunique():,}")
print("\nSplit counts:")
print(df["split"].value_counts().to_string())

# ------------------------- AUTH -------------------------------
print("\n" + "=" * 88)
print("PLANET AUTHENTICATION")
print("=" * 88)
print("Enter your own authorized Planet API key.")
print("The key is hidden and is not written to disk.")
API_KEY = getpass("Planet API key: ").strip()
if not API_KEY:
    raise ValueError("No Planet API key supplied.")

# Data API uses Basic auth.
DATA_SESSION = requests.Session()
DATA_SESSION.auth = (API_KEY, "")

# Orders API uses api-key header.
ORDER_SESSION = requests.Session()
ORDER_SESSION.headers.update({
    "Authorization": f"api-key {API_KEY}",
    "Content-Type": "application/json",
    "X-Planet-App": "Malawi-Landslide-Dataset-Exact-Patch-Rebuilder"
})

# Validate Data API.
_auth_test = DATA_SESSION.post(
    DATA_QUICK_SEARCH,
    json={
        "item_types": ["PSScene"],
        "filter": {
            "type": "DateRangeFilter",
            "field_name": "acquired",
            "config": {
                "gte": "2023-01-01T00:00:00Z",
                "lte": "2023-01-02T00:00:00Z"
            }
        }
    },
    timeout=60
)
if _auth_test.status_code not in (200,):
    raise RuntimeError(
        f"Planet Data API authentication failed (HTTP {_auth_test.status_code}).\n"
        f"{_auth_test.text[:500]}"
    )
print("Planet authentication successful.")

# ------------------------- HELPERS ----------------------------
to_wgs84 = Transformer.from_crs(TARGET_CRS, "EPSG:4326", always_xy=True)

def exact_patch_geometry(row):
    left, bottom, right, top = map(float, (row.left, row.bottom, row.right, row.top))
    xy = [
        (left, bottom), (right, bottom), (right, top),
        (left, top), (left, bottom)
    ]
    ll = [list(to_wgs84.transform(x, y)) for x, y in xy]
    return {"type": "Polygon", "coordinates": [ll]}

def parse_dates(row):
    dates = []

    def add_date(v):
        if v is None or (isinstance(v, float) and np.isnan(v)):
            return
        for m in re.findall(r"\d{4}-\d{2}-\d{2}|\d{8}", str(v)):
            d = m if "-" in m else f"{m[:4]}-{m[4:6]}-{m[6:8]}"
            if d not in dates:
                dates.append(d)

    add_date(row.get("post_date"))

    if "all_intersecting_dates" in row.index:
        add_date(row.get("all_intersecting_dates"))

    # Optional controlled tolerance around primary date.
    if SEARCH_DAY_TOLERANCE > 0 and dates:
        primary = datetime.strptime(dates[0], "%Y-%m-%d")
        for delta in range(-SEARCH_DAY_TOLERANCE, SEARCH_DAY_TOLERANCE + 1):
            d = (primary + timedelta(days=delta)).strftime("%Y-%m-%d")
            if d not in dates:
                dates.append(d)

    return dates

def parse_manifest_ids(row):
    ids = []

    def add_id(v):
        if v is None or (isinstance(v, float) and np.isnan(v)):
            return
        # Planet PSScene IDs of the form YYYYMMDD_HHMMSS_XX_XXXX
        for m in re.findall(r"\d{8}_\d{6}_\d{2}_[A-Za-z0-9]+", str(v)):
            if m not in ids:
                ids.append(m)

    add_id(row.get("post_item_id"))
    if "all_intersecting_items" in row.index:
        add_id(row.get("all_intersecting_items"))
    return ids

def search_planet(aoi, date_string):
    """
    Search only PSScene items that ACTUALLY intersect the exact patch AOI
    during one UTC calendar date.
    """
    d0 = datetime.strptime(date_string, "%Y-%m-%d")
    d1 = d0 + timedelta(days=1)

    payload = {
        "item_types": ["PSScene"],
        "geometry": aoi,
        "filter": {
            "type": "AndFilter",
            "config": [
                {
                    "type": "GeometryFilter",
                    "field_name": "geometry",
                    "config": aoi
                },
                {
                    "type": "DateRangeFilter",
                    "field_name": "acquired",
                    "config": {
                        "gte": d0.strftime("%Y-%m-%dT00:00:00Z"),
                        "lt": d1.strftime("%Y-%m-%dT00:00:00Z")
                    }
                },
                {
                    "type": "PermissionFilter",
                    "config": ["assets:download"]
                }
            ]
        }
    }

    r = DATA_SESSION.post(
        DATA_QUICK_SEARCH + "?_page_size=250&_sort=acquired%20asc",
        json=payload,
        timeout=90
    )
    r.raise_for_status()
    return r.json().get("features", [])

def resolve_items(row):
    """
    Resolve source IDs using Planet's actual catalog geometry.

    Priority:
      A) manifest-listed IDs IF Planet confirms they intersect the patch;
      B) otherwise all downloadable catalog items intersecting the exact
         patch on the required manifest date(s).

    Multiple true overlaps are retained for compositing rather than
    arbitrarily forcing one scene.
    """
    aoi = exact_patch_geometry(row)
    wanted_ids = parse_manifest_ids(row)
    dates = parse_dates(row)

    found = {}
    for date in dates:
        for feat in search_planet(aoi, date):
            found[feat["id"]] = feat

    if not found:
        return [], [], "NO_CATALOG_INTERSECTION"

    preferred = [i for i in wanted_ids if i in found]

    if preferred:
        selected = preferred
        mode = "manifest_id_verified"
    else:
        # Keep all true intersecting items. Sort chronologically so a
        # composite has deterministic ordering.
        selected = sorted(
            found,
            key=lambda i: found[i].get("properties", {}).get("acquired", "")
        )
        mode = "catalog_intersection_fallback"

    acquired = [
        found[i].get("properties", {}).get("acquired", "")
        for i in selected
    ]
    return selected, acquired, mode

def output_names(patch_file):
    name = Path(str(patch_file)).name
    if not name.startswith("RGB_"):
        raise ValueError(f"Unexpected patch filename: {name}")
    suffix = name[4:]
    return f"RGB_{suffix}", f"NIR_{suffix}", f"NDVI_{suffix}"

# ------------------------- ORDER HELPERS ----------------------
def make_order_payload(item_ids, aoi, bundle, order_name):
    tools = [{"clip": {"aoi": aoi}}]
    if len(item_ids) > 1:
        # Later item is applied over earlier item. IDs are passed in
        # deterministic acquisition order.
        tools.append({"composite": {}})

    return {
        "name": order_name,
        "source_type": "scenes",
        "order_type": "full",
        "products": [{
            "item_ids": item_ids,
            "item_type": ITEM_TYPE,
            "product_bundle": bundle
        }],
        "tools": tools
    }

def create_patch_order(item_ids, aoi, patch_key):
    """
    Prefer 4-band Surface Reflectance because the original workflow
    used RGB+NIR. Fall back to 8-band SR when needed.
    """
    attempts = []
    for bundle in ("analytic_sr_udm2", "analytic_8b_sr_udm2"):
        name = f"malawi_{patch_key}_{uuid.uuid4().hex[:7]}"
        payload = make_order_payload(item_ids, aoi, bundle, name)
        r = ORDER_SESSION.post(ORDERS_URL, json=payload, timeout=90)
        if r.status_code in (200, 201, 202):
            return r.json(), bundle
        attempts.append(f"{bundle}: HTTP {r.status_code} {r.text[:400]}")

    raise RuntimeError(
        "Planet rejected the clipped order.\n" + "\n".join(attempts)
    )

def wait_order(order_id, poll=5, timeout_minutes=30):
    deadline = time.time() + timeout_minutes * 60
    while time.time() < deadline:
        r = ORDER_SESSION.get(f"{ORDERS_URL}/{order_id}", timeout=60)
        r.raise_for_status()
        data = r.json()
        state = str(data.get("state", "")).lower()
        if state == "success":
            return data
        if state in ("failed", "cancelled", "partial"):
            raise RuntimeError(f"Order {order_id} finished as '{state}'.")
        time.sleep(poll)
    raise TimeoutError(f"Order {order_id} timed out.")

def find_imagery_result(order_json):
    results = order_json.get("_links", {}).get("results", [])
    candidates = []
    for result in results:
        name = result.get("name", "")
        url = result.get("location")
        low = name.lower()
        if url and (low.endswith(".tif") or low.endswith(".tiff")) and "udm" not in low:
            candidates.append((name, url))

    if not candidates:
        raise RuntimeError("No analytic/composite imagery GeoTIFF found in order results.")

    # Composite output first, otherwise analytic SR.
    candidates.sort(
        key=lambda x: (
            "composite" not in x[0].lower(),
            "analytic" not in x[0].lower(),
            "_sr" not in x[0].lower(),
            x[0]
        )
    )
    return candidates[0]

def download_file(url, destination):
    destination.parent.mkdir(parents=True, exist_ok=True)
    temp = destination.with_suffix(destination.suffix + ".part")
    with ORDER_SESSION.get(url, stream=True, timeout=300) as r:
        r.raise_for_status()
        with open(temp, "wb") as f:
            for chunk in r.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    f.write(chunk)
    temp.replace(destination)

def get_patch_sr(row, item_ids):
    patch_key = Path(str(row.patch_file)).stem
    # Include resolved IDs in cache signature so provenance changes do
    # not silently reuse a previous wrong clip.
    signature = "_".join(i[-6:] for i in item_ids)
    folder = CACHE_DIR / row.split / f"{patch_key}__{signature}"
    folder.mkdir(parents=True, exist_ok=True)

    existing = sorted(folder.glob("*.tif"))
    if existing:
        return existing[0], "cached"

    aoi = exact_patch_geometry(row)
    created, bundle = create_patch_order(item_ids, aoi, patch_key)
    completed = wait_order(created["id"])
    name, url = find_imagery_result(completed)

    destination = folder / (Path(name).name or f"{patch_key}_clip.tif")
    download_file(url, destination)
    return destination, bundle

# ------------------------- UINT8 + GRID -----------------------
def source_band_numbers(count):
    if count == 4:       # Blue, Green, Red, NIR
        return 1, 2, 3, 4
    if count == 8:       # Coastal, Blue, Green-I, Green, Yellow, Red, RedEdge, NIR
        return 2, 4, 6, 8
    raise ValueError(f"Unsupported PlanetScope band count: {count}")

def robust_normalize_to_uint8(arr, p_low=2, p_high=98):
    """
    EXACT normalization used when the original DL dataset was created.

    arr shape: (bands, height, width)
    - Each band is normalized independently.
    - Valid pixels are band > 0.
    - 2nd and 98th percentiles are computed on the FINAL 128x128 patch.
    - Values are clipped to [0,1].
    - Multiplication by 255 is followed by astype(uint8), i.e. truncation,
      exactly as in the original dataset-generation code.
    """
    arr = arr.astype("float32")
    out = np.zeros_like(arr, dtype="uint8")

    for b in range(arr.shape[0]):
        band = arr[b]
        valid = band[band > 0]

        if valid.size == 0:
            continue

        low, high = np.percentile(valid, [p_low, p_high])
        norm = np.clip((band - low) / (high - low + 1e-6), 0, 1)
        out[b] = (norm * 255).astype("uint8")

    return out

def rebuild_outputs(row, clipped_sr, resolved_ids, acquired):
    rgb_name, nir_name, ndvi_name = output_names(row.patch_file)
    base = OUTPUT_DATASET_ROOT / row.split
    rgb_out = base / "RGB" / rgb_name
    nir_out = base / "NIR" / nir_name
    ndvi_out = base / "NDVI" / ndvi_name

    if (
        not OVERWRITE
        and rgb_out.exists()
        and nir_out.exists()
        and ndvi_out.exists()
    ):
        return "already_exists"

    left, bottom, right, top = map(float, (row.left, row.bottom, row.right, row.top))

    if abs((right-left)/TARGET_RESOLUTION - TARGET_WIDTH) > 0.01:
        raise ValueError(f"Wrong manifest width for {row.patch_file}")
    if abs((top-bottom)/TARGET_RESOLUTION - TARGET_HEIGHT) > 0.01:
        raise ValueError(f"Wrong manifest height for {row.patch_file}")

    target_transform = from_origin(
        left, top, TARGET_RESOLUTION, TARGET_RESOLUTION
    )

    with rasterio.open(clipped_sr) as src:
        b, g, r, n = source_band_numbers(src.count)
        data = {}

        for key, band in (("B", b), ("G", g), ("R", r), ("NIR", n)):
            dst = np.zeros((TARGET_HEIGHT, TARGET_WIDTH), dtype=np.float32)
            reproject(
                source=rasterio.band(src, band),
                destination=dst,
                src_transform=src.transform,
                src_crs=src.crs,
                src_nodata=src.nodata,
                dst_transform=target_transform,
                dst_crs=TARGET_CRS,
                dst_nodata=0,
                resampling=Resampling.nearest
            )
            data[key] = dst

    blue, green, red, nir = data["B"], data["G"], data["R"], data["NIR"]

    # IMPORTANT: reproduce the ORIGINAL dataset-generation workflow exactly.
    # The original mosaic bands were ordered B,G,R,NIR; after extracting the
    # final 128x128 window, RGB was reordered to R,G,B and each output band
    # received its own 2nd-98th percentile stretch.
    rgb_raw = np.stack([red, green, blue], axis=0)
    nir_raw = np.stack([nir], axis=0)

    rgb8 = robust_normalize_to_uint8(rgb_raw)
    nir8 = robust_normalize_to_uint8(nir_raw)[0]

    # NDVI is calculated from the un-stretched spectral values.
    # This keeps NDVI physically meaningful before uint8 encoding.
    valid = (nir > 0) & (red > 0)
    denom = nir + red
    ndvi_valid = valid & (denom != 0)
    ndvi = np.zeros_like(nir, dtype=np.float32)
    ndvi[ndvi_valid] = (nir[ndvi_valid] - red[ndvi_valid]) / denom[ndvi_valid]
    ndvi = np.clip(ndvi, -1.0, 1.0)

    # 0 = nodata; 1..255 linearly represents NDVI -1..+1.
    ndvi8 = np.zeros_like(nir8, dtype=np.uint8)
    ndvi8[ndvi_valid] = np.rint(
        1.0 + ((ndvi[ndvi_valid] + 1.0) / 2.0) * 254.0
    ).astype(np.uint8)

    profile = {
        "driver": "GTiff",
        "height": TARGET_HEIGHT,
        "width": TARGET_WIDTH,
        "crs": TARGET_CRS,
        "transform": target_transform,
        "dtype": "uint8",
        "compress": "LZW",
        "tiled": True,
        "nodata": 0,
    }

    common_tags = {
        "source": "PlanetScope Surface Reflectance",
        "source_item_ids": ";".join(resolved_ids),
        "source_acquired_utc": ";".join(acquired),
        "target_crs": TARGET_CRS,
        "target_resolution_m": str(TARGET_RESOLUTION),
    }

    with rasterio.open(rgb_out, "w", count=3, **profile) as dst:
        dst.write(rgb8)
        dst.set_band_description(1, "Red")
        dst.set_band_description(2, "Green")
        dst.set_band_description(3, "Blue")
        dst.update_tags(**common_tags, uint8_conversion="per-band 2nd-98th percentile stretch on final 128x128 patch")

    with rasterio.open(nir_out, "w", count=1, **profile) as dst:
        dst.write(nir8, 1)
        dst.set_band_description(1, "NIR")
        dst.update_tags(**common_tags, uint8_conversion="per-band 2nd-98th percentile stretch on final 128x128 patch")

    with rasterio.open(ndvi_out, "w", count=1, **profile) as dst:
        dst.write(ndvi8, 1)
        dst.set_band_description(1, "NDVI_uint8")
        dst.update_tags(
            **common_tags,
            ndvi_formula="(NIR-Red)/(NIR+Red)",
            uint8_encoding="0=nodata;1..255 maps NDVI -1..+1"
        )

    # Strict geometry/datatype verification.
    expected_bounds = (left, bottom, right, top)
    for path, bands in ((rgb_out, 3), (nir_out, 1), (ndvi_out, 1)):
        with rasterio.open(path) as chk:
            actual_bounds = (
                chk.bounds.left, chk.bounds.bottom,
                chk.bounds.right, chk.bounds.top
            )
            if (chk.width, chk.height) != (TARGET_WIDTH, TARGET_HEIGHT):
                raise RuntimeError(f"Size check failed: {path}")
            if chk.count != bands:
                raise RuntimeError(f"Band check failed: {path}")
            if chk.dtypes[0] != "uint8":
                raise RuntimeError(f"uint8 check failed: {path}")
            if chk.crs is None or chk.crs.to_epsg() != 32736:
                raise RuntimeError(f"CRS check failed: {path}")
            if any(abs(a-b) > 1e-6 for a, b in zip(actual_bounds, expected_bounds)):
                raise RuntimeError(f"Bounds check failed: {path}")

    return "created"

# ------------------------- PROCESS ----------------------------
records = []
provenance = []

print("\n" + "=" * 88)
print("VERIFYING PLANET CATALOG INTERSECTIONS + DOWNLOADING ONLY PATCH AOIs")
print("=" * 88)
print("No full-scene download is requested by this notebook.\n")

for _, row in tqdm(df.iterrows(), total=len(df), desc="Exact PlanetScope patches"):
    rec = {
        "split": row.split,
        "patch_file": row.patch_file,
        "manifest_post_date": row.post_date,
        "manifest_post_item_id": row.post_item_id,
    }

    try:
        rgb_name, nir_name, ndvi_name = output_names(row.patch_file)
        base = OUTPUT_DATASET_ROOT / row.split

        # Even when outputs already exist, provenance is resolved so the
        # verification CSV remains complete.
        resolved_ids, acquired, resolve_mode = resolve_items(row)

        if not resolved_ids:
            raise RuntimeError(
                "No authorized PSScene catalog item intersects this exact "
                "patch footprint on the manifest post-event date(s)."
            )

        provenance.append({
            "split": row.split,
            "patch_file": row.patch_file,
            "left": row.left,
            "bottom": row.bottom,
            "right": row.right,
            "top": row.top,
            "manifest_post_date": row.post_date,
            "manifest_post_item_id": row.post_item_id,
            "verified_item_ids": ";".join(resolved_ids),
            "verified_acquired_utc": ";".join(acquired),
            "number_verified_items": len(resolved_ids),
            "resolution_mode": resolve_mode,
        })

        if (
            not OVERWRITE
            and (base/"RGB"/rgb_name).exists()
            and (base/"NIR"/nir_name).exists()
            and (base/"NDVI"/ndvi_name).exists()
        ):
            status = "already_exists"
            delivery = "not_needed"
        else:
            clipped_sr, delivery = get_patch_sr(row, resolved_ids)
            status = rebuild_outputs(row, clipped_sr, resolved_ids, acquired)

        rec.update(
            verified_item_ids=";".join(resolved_ids),
            verified_acquired_utc=";".join(acquired),
            number_verified_items=len(resolved_ids),
            resolution_mode=resolve_mode,
            delivery=delivery,
            status=status,
            error=""
        )

    except Exception as e:
        rec.update(
            verified_item_ids="",
            verified_acquired_utc="",
            number_verified_items=0,
            resolution_mode="",
            delivery="",
            status="FAILED",
            error=str(e)
        )
        print(f"\nFAILED: {row.patch_file}\n{e}")

    records.append(rec)

    # Resume-safe logs.
    pd.DataFrame(records).to_csv(LOG_PATH, index=False, encoding="utf-8-sig")
    if provenance:
        pd.DataFrame(provenance).to_csv(PROVENANCE_PATH, index=False, encoding="utf-8-sig")

# ------------------------- REPORT -----------------------------
log = pd.DataFrame(records)

print("\n" + "=" * 88)
print("FINAL REPORT")
print("=" * 88)
print("\nStatus:")
print(log["status"].value_counts(dropna=False).to_string())

if "resolution_mode" in log:
    print("\nPlanet item resolution:")
    print(log["resolution_mode"].replace("", np.nan).value_counts(dropna=False).to_string())

print("\nGenerated GeoTIFF counts:")
for split in ("train", "val", "test"):
    print(f"\n{split.upper()}")
    for modality in ("RGB", "NIR", "NDVI"):
        folder = OUTPUT_DATASET_ROOT / split / modality
        print(f"  {modality:<4}: {len(list(folder.glob('*.tif'))):,}")

failed = log[log["status"] == "FAILED"]
print("\nReconstruction log :", LOG_PATH)
print("Verified provenance:", PROVENANCE_PATH)

if len(failed):
    print(f"\nWARNING: {len(failed):,} patch(es) failed.")
    print("Inspect their 'error' field in the reconstruction log.")
else:
    print("\nSUCCESS: every manifest patch was reconstructed.")

print("\nOUTPUT SPECIFICATION")
print("  RGB  : uint8, 3 bands (R,G,B), original 2-98 percentile normalization")
print("  NIR  : uint8, 1 band, original 2-98 percentile normalization")
print("  NDVI : uint8, 1 band; 0=NoData; 1..255 represents -1..+1")
print("  CRS  :", TARGET_CRS)
print("  Pixel:", f"{TARGET_RESOLUTION} m")
print("  Size :", f"{TARGET_WIDTH} x {TARGET_HEIGHT}")
print("  Bounds/transform: exactly reconstructed from each CSV row")
print("\nFinished.")
